In [1]:
import pandas as pd
import numpy as np


In [2]:
sales_long = pd.read_csv(
    "../data/processed/model_data.csv",
    parse_dates=["date"]
)

sales_long.info()
sales_long.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_3436\3380743162.py:1: DtypeWarning: Columns (16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_long = pd.read_csv(


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5832737 entries, 0 to 5832736
Data columns (total 22 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            object        
 1   item_id       object        
 2   dept_id       object        
 3   cat_id        object        
 4   store_id      object        
 5   state_id      object        
 6   d             object        
 7   sales         int64         
 8   date          datetime64[ns]
 9   wm_yr_wk      int64         
 10  weekday       object        
 11  wday          int64         
 12  month         int64         
 13  year          int64         
 14  event_name_1  object        
 15  event_type_1  object        
 16  event_name_2  object        
 17  event_type_2  object        
 18  snap_CA       int64         
 19  snap_TX       int64         
 20  snap_WI       int64         
 21  sell_price    float64       
dtypes: datetime64[ns](1), float64(1), int64(8), object(12)
memory usag

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
2,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,2.0
3,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,2,2011,NaN,NaN,NaN,NaN,1,1,0,2.0
4,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,2,2011,NaN,NaN,NaN,NaN,1,0,1,2.0


In [3]:
sales_long.isnull().sum().sort_values(ascending=False).head(10)

event_type_2    5820541
event_name_2    5820541
event_type_1    5363191
event_name_1    5363191
sell_price      1129842
id                    0
state_id              0
store_id              0
cat_id                0
dept_id               0
dtype: int64

We know that weekends, month start or ends can affect the sales, also we have to know the weekly trends and quarterly trends of the sale in the store


In [4]:
sales_long['day'] = sales_long['date'].dt.day

sales_long['week'] = sales_long['date'].dt.isocalendar().week.astype("int16")

sales_long["quarter"] = sales_long['date'].dt.quarter

sales_long['is_weekend'] = (
    sales_long["weekday"]
    .isin(["Saturday", "Sunday"])
    .astype("int8")
)

sales_long[
    [
        "date",
        "day",
        "week",
        "month",
        "quarter",
        "weekday",
        "is_weekend",
    ]
].head(10)

,date,day,week,month,quarter,weekday,is_weekend
0,2011-01-29,29,4,1,1,Saturday,1
1,2011-01-30,30,4,1,1,Sunday,1
2,2011-01-31,31,5,1,1,Monday,0
3,2011-02-01,1,5,2,1,Tuesday,0
4,2011-02-02,2,5,2,1,Wednesday,0
5,2011-02-03,3,5,2,1,Thursday,0
6,2011-02-04,4,5,2,1,Friday,0
7,2011-02-05,5,5,2,1,Saturday,1
8,2011-02-06,6,5,2,1,Sunday,1
9,2011-02-07,7,6,2,1,Monday,0


In [5]:
# Calendar-derived features (if not already extracted into numeric form):
# day_of_week
# week_of_year
# is_weekend
# quarter
# cyclic encoding (optional)
# Event features (event_name_*, event_type_*)
# SNAP features (snap_CA, snap_TX, snap_WI)
# Price-derived features (price change, normalized price, etc.)
# Preparing the final feature matrix
# Train/validation split for time series
# Baseline forecasting model
# Model evaluation
# Inventory optimization layer (safety stock, reorder point, EOQ)



Working on Price features to make the model learn the price sales relation


Handling the Mising Values

In [6]:
missing = sales_long.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(missing)

missing_percentage = (
    sales_long
    .isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_percentage[missing_percentage > 0]

event_type_2    5820541
event_name_2    5820541
event_type_1    5363191
event_name_1    5363191
sell_price      1129842
dtype: int64


event_type_2    99.790904
event_name_2    99.790904
event_type_1    91.949817
event_name_1    91.949817
sell_price      19.370700
dtype: float64

In [7]:
# so filling the event columns with "No Event" where there is no event present so to minimise the number of null values in them
event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]

sales_long[event_cols] = sales_long[event_cols].fillna("No_Event")

In [8]:
print(sales_long["sell_price"].isna().mean())


items_with_missing_price = sales_long.loc[
    sales_long["sell_price"].isna(),
    "item_id"
].unique()

print(f"Number of items: {len(items_with_missing_price)}")
print(items_with_missing_price)


sales_long.loc[
    sales_long["sell_price"].isna(),
    ["item_id", "date", "wm_yr_wk"]
]



0.19370700239013006
Number of items: 1845
['FOODS_1_004' 'FOODS_1_008' 'FOODS_1_009' ... 'HOUSEHOLD_2_510'
 'HOUSEHOLD_2_513' 'HOUSEHOLD_2_515']


,item_id,date,wm_yr_wk
5739,FOODS_1_004,2011-01-29,11101
5740,FOODS_1_004,2011-01-30,11101
5741,FOODS_1_004,2011-01-31,11101
5742,FOODS_1_004,2011-02-01,11101
5743,FOODS_1_004,2011-02-02,11101
...,...,...,...
5829977,HOUSEHOLD_2_515,2013-12-30,11349
5829978,HOUSEHOLD_2_515,2013-12-31,11349
5829979,HOUSEHOLD_2_515,2014-01-01,11349
5829980,HOUSEHOLD_2_515,2014-01-02,11349


In [9]:
sales_long[["item_id", "date", "wm_yr_wk", "sell_price"]]

,item_id,date,wm_yr_wk,sell_price
0,FOODS_1_001,2011-01-29,11101,2.00
1,FOODS_1_001,2011-01-30,11101,2.00
2,FOODS_1_001,2011-01-31,11101,2.00
3,FOODS_1_001,2011-02-01,11101,2.00
4,FOODS_1_001,2011-02-02,11101,2.00
...,...,...,...,...
5832732,HOUSEHOLD_2_516,2016-04-20,11612,5.94
5832733,HOUSEHOLD_2_516,2016-04-21,11612,5.94
5832734,HOUSEHOLD_2_516,2016-04-22,11612,5.94
5832735,HOUSEHOLD_2_516,2016-04-23,11613,5.94


In [10]:
# to confirm whether the missing prices are due to introducing the item early on and not opening it for sale
# to test this hypothesis, first I did a try to find whether this item has a price at all by checking its tail of last 20 sales days it made
# then checking whether the item was sold on a day but the sell_price is not filled
# if they all are true then our assumption is correct
# checking for the product "FOODS_1_004"
print(sales_long.loc[
    sales_long["item_id"] == "FOODS_1_004",
    ["date", "sell_price"]
].tail(20))

print(sales_long.loc[
    (sales_long["item_id"] == "FOODS_1_004") &
    (sales_long["sell_price"].isna()),
    ["date", "sales"]
])

sales_long.loc[
    (sales_long["item_id"] == "FOODS_1_004") &
    (sales_long["sell_price"].isna()) &
    (sales_long["sales"] > 0),
    ["date", "sales"]
]

           date  sell_price
7632 2016-04-05        1.96
7633 2016-04-06        1.96
7634 2016-04-07        1.96
7635 2016-04-08        1.96
7636 2016-04-09        1.96
7637 2016-04-10        1.96
7638 2016-04-11        1.96
7639 2016-04-12        1.96
7640 2016-04-13        1.96
7641 2016-04-14        1.96
7642 2016-04-15        1.96
7643 2016-04-16        1.96
7644 2016-04-17        1.96
7645 2016-04-18        1.96
7646 2016-04-19        1.96
7647 2016-04-20        1.96
7648 2016-04-21        1.96
7649 2016-04-22        1.96
7650 2016-04-23        1.96
7651 2016-04-24        1.96
           date  sales
5739 2011-01-29      0
5740 2011-01-30      0
5741 2011-01-31      0
5742 2011-02-01      0
5743 2011-02-02      0
...         ...    ...
6133 2012-02-27      0
6134 2012-02-28      0
6135 2012-02-29      0
6136 2012-03-01      0
6137 2012-03-02      0

[399 rows x 2 columns]


,date,sales


In [11]:
sales_long.groupby("item_id")["sell_price"].apply(lambda x: x.notna().sum()).describe()

count    3049.000000
mean     1542.438504
std       461.588467
min        72.000000
25%      1164.000000
50%      1850.000000
75%      1913.000000
max      1913.000000
Name: sell_price, dtype: float64

In [12]:
# Here I am checking whether the rows having sales_price = 0 have sales > 0
# if not then we can just remove all the rows having sale_price = 0

print(sales_long.loc[
    sales_long["sell_price"].isna(),
    "sales"
].describe())

print(sales_long.loc[
    sales_long["sell_price"].isna(),
    "sales"
].value_counts().head(10))

count    1129842.0
mean           0.0
std            0.0
min            0.0
25%            0.0
50%            0.0
75%            0.0
max            0.0
Name: sales, dtype: float64
sales
0    1129842
Name: count, dtype: int64


In [13]:
# Dropping the rows where sell_price = 0 as there is no benefit of having info about the product which is either not ready for sale at that time or was stopped selling by the store and so sales are also 0 for it that time
# Our model will not learn anything from it and filling these values also give false info to our model which is bad

sales_long = sales_long[sales_long["sell_price"].notna()].copy()
sales_long.reset_index(drop=True, inplace=True)

In [14]:
sales_long["lag_1"] = (
    sales_long.groupby("item_id")["sales"]
    .shift(1)
)

sales_long["lag_7"] = (
    sales_long.groupby("item_id")["sales"]
    .shift(7)
)

sales_long["lag_28"] = (
    sales_long.groupby("item_id")["sales"]
    .shift(28)
)

In [15]:
sales_long["rolling_mean_7"] = (
    sales_long.groupby("item_id")["sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

sales_long["rolling_mean_28"] = (
    sales_long.groupby("item_id")["sales"]
    .transform(lambda x: x.shift(1).rolling(28).mean())
)

sales_long["rolling_std_7"] = (
    sales_long.groupby("item_id")["sales"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

In [16]:
sales_long["price_change"] = (
    sales_long.groupby("item_id")["sell_price"]
    .pct_change()
)

sales_long["price_max"] = (
    sales_long.groupby("item_id")["sell_price"]
    .transform("max")
)

sales_long["price_min"] = (
    sales_long.groupby("item_id")["sell_price"]
    .transform("min")
)

sales_long["price_norm"] = (
    sales_long["sell_price"] / sales_long["price_max"]
)

sales_long["price_range"] = (
    sales_long["price_max"] - sales_long["price_min"]
)

In [17]:
missing = (
    sales_long.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

rolling_mean_28    85372
lag_28             85372
lag_7              21343
rolling_mean_7     21343
rolling_std_7      21343
lag_1               3049
price_change        3049
dtype: int64

In [18]:
# filling the price_change NA as 0 because they tell there is no last price to compare with, so puting 0 can work here

sales_long["price_change"] = sales_long["price_change"].fillna(0)


# dropping the rows where lag, rolling features have value NA
model_data = sales_long.dropna().reset_index(drop=True)

In [19]:
missing = (
    model_data.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

Series([], dtype: int64)

In [20]:
categorical_cols = model_data.select_dtypes(include="object").columns
print(categorical_cols)
print(f"\nTotal categorical columns: {len(categorical_cols)}")

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'weekday', 'event_name_1', 'event_type_1', 'event_name_2',
       'event_type_2'],
      dtype='object')

Total categorical columns: 12


In [21]:
print(model_data["store_id"].nunique())
print(model_data["state_id"].nunique())

print(model_data["store_id"].unique())
print(model_data["state_id"].unique())

model_data.drop(columns=["store_id", "state_id"], inplace=True)

1
1
['CA_1']
['CA']


In [22]:
constant_cols = [col for col in model_data.columns if model_data[col].nunique() == 1]

print(constant_cols)

[]


In [23]:
categorical_cols = model_data.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n{col}")
    print(f"Unique values: {model_data[col].nunique()}")
    print(model_data[col].unique()[:10])  # Show first 10 unique values


id
Unique values: 3049
['FOODS_1_001_CA_1_validation' 'FOODS_1_002_CA_1_validation'
 'FOODS_1_003_CA_1_validation' 'FOODS_1_004_CA_1_validation'
 'FOODS_1_005_CA_1_validation' 'FOODS_1_006_CA_1_validation'
 'FOODS_1_008_CA_1_validation' 'FOODS_1_009_CA_1_validation'
 'FOODS_1_010_CA_1_validation' 'FOODS_1_011_CA_1_validation']

item_id
Unique values: 3049
['FOODS_1_001' 'FOODS_1_002' 'FOODS_1_003' 'FOODS_1_004' 'FOODS_1_005'
 'FOODS_1_006' 'FOODS_1_008' 'FOODS_1_009' 'FOODS_1_010' 'FOODS_1_011']

dept_id
Unique values: 7
['FOODS_1' 'FOODS_2' 'FOODS_3' 'HOBBIES_1' 'HOBBIES_2' 'HOUSEHOLD_1'
 'HOUSEHOLD_2']

cat_id
Unique values: 3
['FOODS' 'HOBBIES' 'HOUSEHOLD']

d
Unique values: 1885
['d_29' 'd_30' 'd_31' 'd_32' 'd_33' 'd_34' 'd_35' 'd_36' 'd_37' 'd_38']

weekday
Unique values: 7
['Saturday' 'Sunday' 'Monday' 'Tuesday' 'Wednesday' 'Thursday' 'Friday']

event_name_1
Unique values: 31
['No_Event' 'LentStart' 'LentWeek2' 'StPatricksDay' 'Purim End'
 'OrthodoxEaster' 'Pesach End' 'Cinco De

In [24]:
from sklearn.preprocessing import LabelEncoder

item_encoder = LabelEncoder()

model_data["item_id"] = item_encoder.fit_transform(model_data["item_id"])

In [25]:
import joblib

joblib.dump(item_encoder, "../models/item_encoder.pkl")

['../models/item_encoder.pkl']

In [26]:
model_data = pd.get_dummies(
    model_data,
    columns=["dept_id"],
    prefix="dept",
    dtype=int
)

model_data.head()

,id,item_id,cat_id,d,sales,date,wm_yr_wk,weekday,wday,month,...,price_min,price_norm,price_range,dept_FOODS_1,dept_FOODS_2,dept_FOODS_3,dept_HOBBIES_1,dept_HOBBIES_2,dept_HOUSEHOLD_1,dept_HOUSEHOLD_2
0,FOODS_1_001_CA_1_validation,0,FOODS,d_29,2,2011-02-26,11105,Saturday,1,2,...,2.0,0.892857,0.24,1,0,0,0,0,0,0
1,FOODS_1_001_CA_1_validation,0,FOODS,d_30,2,2011-02-27,11105,Sunday,2,2,...,2.0,0.892857,0.24,1,0,0,0,0,0,0
2,FOODS_1_001_CA_1_validation,0,FOODS,d_31,0,2011-02-28,11105,Monday,3,2,...,2.0,0.892857,0.24,1,0,0,0,0,0,0
3,FOODS_1_001_CA_1_validation,0,FOODS,d_32,2,2011-03-01,11105,Tuesday,4,3,...,2.0,0.892857,0.24,1,0,0,0,0,0,0
4,FOODS_1_001_CA_1_validation,0,FOODS,d_33,1,2011-03-02,11105,Wednesday,5,3,...,2.0,0.892857,0.24,1,0,0,0,0,0,0


In [27]:
model_data = pd.get_dummies(
    model_data,
    columns=["cat_id"],
    prefix="cat",
    dtype=int
)

event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

model_data = pd.get_dummies(
    model_data,
    columns=event_cols,
    dtype=int
)

In [28]:
import numpy as np

weekday_map = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6
}

model_data["weekday_num"] = model_data["weekday"].map(weekday_map)

model_data["weekday_sin"] = np.sin(2 * np.pi * model_data["weekday_num"] / 7)
model_data["weekday_cos"] = np.cos(2 * np.pi * model_data["weekday_num"] / 7)

model_data.drop(columns=["weekday", "weekday_num"], inplace=True)

model_data.head()

,id,item_id,d,sales,date,wm_yr_wk,wday,month,year,snap_CA,...,event_name_2_Cinco De Mayo,event_name_2_Easter,event_name_2_Father's day,event_name_2_No_Event,event_name_2_OrthodoxEaster,event_type_2_Cultural,event_type_2_No_Event,event_type_2_Religious,weekday_sin,weekday_cos
0,FOODS_1_001_CA_1_validation,0,d_29,2,2011-02-26,11105,1,2,2011,0,...,0,0,0,1,0,0,1,0,-0.974928,-0.222521
1,FOODS_1_001_CA_1_validation,0,d_30,2,2011-02-27,11105,2,2,2011,0,...,0,0,0,1,0,0,1,0,-0.781831,0.623490
2,FOODS_1_001_CA_1_validation,0,d_31,0,2011-02-28,11105,3,2,2011,0,...,0,0,0,1,0,0,1,0,0.000000,1.000000
3,FOODS_1_001_CA_1_validation,0,d_32,2,2011-03-01,11105,4,3,2011,1,...,0,0,0,1,0,0,1,0,0.781831,0.623490
4,FOODS_1_001_CA_1_validation,0,d_33,1,2011-03-02,11105,5,3,2011,1,...,0,0,0,1,0,0,1,0,0.974928,-0.222521


In [29]:
model_data.columns

Index(['id', 'item_id', 'd', 'sales', 'date', 'wm_yr_wk', 'wday', 'month',
       'year', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'day', 'week',
       'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7',
       'rolling_mean_28', 'rolling_std_7', 'price_change', 'price_max',
       'price_min', 'price_norm', 'price_range', 'dept_FOODS_1',
       'dept_FOODS_2', 'dept_FOODS_3', 'dept_HOBBIES_1', 'dept_HOBBIES_2',
       'dept_HOUSEHOLD_1', 'dept_HOUSEHOLD_2', 'cat_FOODS', 'cat_HOBBIES',
       'cat_HOUSEHOLD', 'event_name_1_Chanukah End', 'event_name_1_Christmas',
       'event_name_1_Cinco De Mayo', 'event_name_1_ColumbusDay',
       'event_name_1_Easter', 'event_name_1_Eid al-Fitr',
       'event_name_1_EidAlAdha', 'event_name_1_Father's day',
       'event_name_1_Halloween', 'event_name_1_IndependenceDay',
       'event_name_1_LaborDay', 'event_name_1_LentStart',
       'event_name_1_LentWeek2', 'event_name_1_MartinLutherKingDay',
       'event_name_1_Memori

In [30]:

print(model_data.shape)
print(model_data.info())
print(model_data.head())
model_data.to_parquet("../data/processed/preprocessed_data.parquet", index=False)
model_data.to_csv("../data/processed/preprocessed_data.csv", index=False)

print("Preprocessed data saved!")


(4617523, 84)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4617523 entries, 0 to 4617522
Data columns (total 84 columns):
 #   Column                            Dtype         
---  ------                            -----         
 0   id                                object        
 1   item_id                           int64         
 2   d                                 object        
 3   sales                             int64         
 4   date                              datetime64[ns]
 5   wm_yr_wk                          int64         
 6   wday                              int64         
 7   month                             int64         
 8   year                              int64         
 9   snap_CA                           int64         
 10  snap_TX                           int64         
 11  snap_WI                           int64         
 12  sell_price                        float64       
 13  day                               int32         
 14  week